# Avito: анализ данных и схема валидации

Цель раздела — проверить качество полей, сопоставимость train и
benchmark и построить валидацию без пересечения формулировок запросов.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [ ]:
from pathlib import Path
import os

# Для локального запуска достаточно изменить PROJECT_DIR или задать
# AVITO_PROJECT_DIR / AVITO_DATA_DIR / AVITO_OUTPUT_DIR.
KAGGLE_DATA_DIR = Path("/kaggle/input/datasets/yuriymagus/avito-nlp-interns")
DEFAULT_PROJECT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()

PROJECT_DIR = Path(os.environ.get("AVITO_PROJECT_DIR", DEFAULT_PROJECT_DIR)).expanduser().resolve()
DATA_DIR = Path(
    os.environ.get(
        "AVITO_DATA_DIR",
        KAGGLE_DATA_DIR if KAGGLE_DATA_DIR.exists() else PROJECT_DIR / "data",
    )
).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get("AVITO_OUTPUT_DIR", PROJECT_DIR)).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_FILES = (
    "train.parquet",
    "benchmark_queries.parquet",
    "benchmark_items.parquet",
)
missing_files = [name for name in REQUIRED_FILES if not (DATA_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(
        f"Не найдены файлы {missing_files} в {DATA_DIR}. "
        "Измените DATA_DIR в первой ячейке."
    )

print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"DATA_DIR:    {DATA_DIR}")
print(f"OUTPUT_DIR:  {OUTPUT_DIR}")


In [ ]:
train = pd.read_parquet(DATA_DIR / "train.parquet")
queries = pd.read_parquet(DATA_DIR / "benchmark_queries.parquet")
items = pd.read_parquet(DATA_DIR / "benchmark_items.parquet")

print(f"train:   {train.shape}")
print(f"queries: {queries.shape}")
print(f"items:   {items.shape}")

frames = {"train": train, "queries": queries, "items": items}


## 1. Схема и пропуски


In [ ]:
for name, frame in frames.items():
    print(f"\n{name} — типы столбцов")
    display(frame.dtypes.rename("dtype").to_frame())

    missing = pd.DataFrame({
        "missing_count": frame.isna().sum(),
        "missing %": frame.isna().mean().mul(100).round(2),
    })
    display(missing.loc[missing["missing_count"].gt(0)]
            .sort_values("missing_count", ascending=False))


Числовые по смыслу поля сохранены как строки, поэтому перед их
использованием нужна явная конвертация. Отдельно проверим значения,
которые нельзя считать корректными числовыми признаками.


In [ ]:
numeric_rules = {
    "item_price": lambda values: values.lt(0),
    "item_rating": lambda values: values.lt(0) | values.gt(5),
    "item_rating_reviews_count": lambda values: values.lt(0),
    "item_latitude": lambda values: values.lt(-90) | values.gt(90),
    "item_longitude": lambda values: values.lt(-180) | values.gt(180),
}

quality_rows = []
for dataset_name, frame in {"train": train, "items": items}.items():
    for column, invalid_rule in numeric_rules.items():
        raw = frame[column]
        numeric = pd.to_numeric(raw, errors="coerce")
        parse_errors = raw.notna() & numeric.isna()
        invalid_values = numeric.notna() & invalid_rule(numeric)
        quality_rows.append({
            "dataset": dataset_name,
            "column": column,
            "missing": int(raw.isna().sum()),
            "parse_errors": int(parse_errors.sum()),
            "invalid_values": int(invalid_values.sum()),
            "min": numeric.min(),
            "max": numeric.max(),
        })

numeric_quality = pd.DataFrame(quality_rows).set_index(["dataset", "column"])
display(numeric_quality)


Отрицательные цены и другие значения вне допустимых диапазонов считаются
ошибками данных. Итоговый поиск не использует цену и координаты; рейтинг
и число отзывов нужны только для заполнения редких выдач без достаточного
числа текстовых совпадений. Строки не удаляются, а пропущенный текст
далее заменяется пустой строкой.


## 2. Повторяемость запросов и объявлений


Одинаковые тексты запросов не являются дублями: это разные поисковые
события. Нас интересует повторяемость формулировок, поскольку случайное
разбиение строк привело бы к утечке между train и валидацией.


In [ ]:
normalized_query = (
    train["search_query"]
    .astype("string")
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
query_frequency = normalized_query.value_counts()

print(f"Уникальных исходных текстов: {train['search_query'].nunique():,}")
print(f"Уникальных нормализованных текстов: {normalized_query.nunique():,}")
display(
    query_frequency
    .describe(percentiles=[0.5, 0.9, 0.99])
    .rename("Строк из train на уникальный запрос")
    .to_frame()
)
display(query_frequency.head(10).rename("Частота").to_frame())


In [ ]:
frequency_buckets = pd.cut(
    query_frequency,
    bins=[0, 1, 2, 5, 10, 50, 100, float("inf")],
    labels=["1", "2", "3–5", "6–10", "11–50", "51–100", ">100"],
)
bucket_counts = frequency_buckets.value_counts(sort=False)

ax = bucket_counts.plot.bar(figsize=(8, 3.5), rot=0)
ax.set_title("Частота запросов в train")
ax.set_xlabel("Строк train на один текст запроса")
ax.set_ylabel("Число уникальных запросов")
plt.tight_layout()
plt.show()


In [ ]:
item_frequency = train["item_id"].value_counts()

print(f"Уникальных выбранных items в train: {train['item_id'].nunique():,}")
print(f"Уникальных items в benchmark-корпусе: {items['item_id'].nunique():,}")
print(f"Дубликатов item_id в benchmark-корпусе: {items['item_id'].duplicated().sum():,}")
print(f"Дубликатов query_id в benchmark-запросах: {queries['query_id'].duplicated().sum():,}")

display(
    item_frequency
    .describe(percentiles=[0.5, 0.9, 0.99])
    .rename("Строк train на item")
    .to_frame()
)

train_item_ids = set(train["item_id"].astype("string"))
benchmark_item_ids = set(items["item_id"].astype("string"))
overlap = len(train_item_ids & benchmark_item_ids)

print(
    f"Пересечение train items с benchmark-корпусом: "
    f"{overlap:,} / {len(train_item_ids):,} "
    f"({overlap / len(train_item_ids):.2%})"
)


Для повторяющихся `item_id` проверяем согласованность основных полей.
Конфликтов в заголовке, категории и локации обнаружено не было.


In [ ]:
item_fields = [
    "item_id",
    "item_title_raw",
    "item_category_id",
    "item_microcat_id",
    "item_location_id",
]
train_items = train.drop_duplicates("item_id").copy()

consistency = train.groupby("item_id")[
    ["item_title_raw", "item_category_id", "item_location_id"]
].nunique(dropna=False)

print(f"Уникальных выбранных items в train: {len(train_items):,}")
print(f"Items с несколькими title: {consistency['item_title_raw'].gt(1).sum():,}")
print(
    "Items с несколькими category: "
    f"{consistency['item_category_id'].gt(1).sum():,}"
)
print(
    "Items с несколькими location: "
    f"{consistency['item_location_id'].gt(1).sum():,}"
)

microcat_comparison = pd.DataFrame({
    "train_selected_pct": (
        train_items["item_microcat_id"]
        .value_counts(normalize=True)
        .mul(100)
    ),
    "benchmark_corpus_pct": (
        items["item_microcat_id"]
        .value_counts(normalize=True)
        .mul(100)
    ),
}).fillna(0)

microcat_comparison["difference_pp"] = (
    microcat_comparison["benchmark_corpus_pct"]
    - microcat_comparison["train_selected_pct"]
)

display(
    microcat_comparison
    .reindex(microcat_comparison["difference_pp"].abs().sort_values(
        ascending=False
    ).index)
    .head(15)
    .round(2)
)


## 3. Категории, география и длины текстов


In [ ]:
category_summary = pd.DataFrame({
    "train_queries": train["search_category"].value_counts(),
    "benchmark_queries": queries["search_category"].value_counts(),
    "train_items": train["item_category_id"].value_counts(),
    "benchmark_items": items["item_category_id"].value_counts(),
}).fillna(0).astype(int)

display(category_summary.sort_values("benchmark_queries", ascending=False).head(12))

for name, frame in (("train", train), ("benchmark_items", items)):
    print(f"\n{name}: наиболее частые microcategories")
    display(frame["item_microcat_id"].value_counts().head(10).to_frame("count"))

category_match = train["search_category"].eq(train["item_category_id"]).mean()
location_match = train["search_location_id"].eq(train["item_location_id"]).mean()

print(f"\nБуквальное совпадение category в train: {category_match:.2%}")
print(f"Буквальное совпадение location в train: {location_match:.2%}")


In [ ]:
text_columns = [
    ("train", train, "search_query"),
    ("train", train, "item_title_raw"),
    ("train", train, "item_description_raw"),
    ("benchmark_queries", queries, "search_query"),
    ("benchmark_items", items, "item_title_raw"),
    ("benchmark_items", items, "item_description_raw"),
]

length_rows = []
for name, frame, column in text_columns:
    lengths = frame[column].astype("string").str.len().dropna()
    length_rows.append({
        "dataset": name,
        "field": column,
        "median": lengths.median(),
        "p90": lengths.quantile(0.90),
        "p99": lengths.quantile(0.99),
        "max": lengths.max(),
    })

display(pd.DataFrame(length_rows).round(1))


In [ ]:
train_unique_lengths = (
    normalized_query.drop_duplicates().str.len().dropna().astype(int)
)
benchmark_lengths = (
    queries["search_query"].astype("string").str.len().dropna().astype(int)
)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(
    train_unique_lengths,
    bins=range(0, 82, 2),
    density=True,
    alpha=0.55,
    label="Уникальные тексты train",
)
ax.hist(
    benchmark_lengths,
    bins=range(0, 82, 2),
    density=True,
    alpha=0.55,
    label="Benchmark-запросы",
)
ax.set_xlim(0, 80)
ax.set_xlabel("Длина запроса, символов")
ax.set_ylabel("Плотность")
ax.set_title("Длины уникальных формулировок запросов")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Медиана уникальных текстов train: {train_unique_lengths.median():.0f}")
print(f"Медиана benchmark-запросов: {benchmark_lengths.median():.0f}")


In [ ]:
location_diagnostics = train.assign(
    same_location=train["search_location_id"].eq(train["item_location_id"]),
    has_params=(
        train["search_infm_params_text"]
        .astype("string")
        .str.strip()
        .ne("")
    ),
)

display(
    location_diagnostics
    .groupby("has_params")["same_location"]
    .agg(events="size", match_rate="mean")
    .assign(match_rate=lambda df: df["match_rate"].mul(100))
    .round(2)
)

text_diagnostics = items.assign(
    title_length=items["item_title_raw"].astype("string").str.len(),
    description_length=items["item_description_raw"].astype("string").str.len(),
)

print(
    "Объявлений с title длиной ≤5 символов: "
    f"{text_diagnostics['title_length'].le(5).sum():,}"
)
print(
    "Объявлений с description длиной >5 000 символов: "
    f"{text_diagnostics['description_length'].gt(5000).sum():,}"
)

display(
    text_diagnostics
    .nsmallest(5, "title_length")[
        ["item_title_raw", "title_length", "item_microcat_id"]
    ]
)
display(
    text_diagnostics
    .nlargest(5, "description_length")[
        ["item_title_raw", "description_length", "item_microcat_id"]
    ]
)


## 4. Сопоставимость train и benchmark


In [ ]:
def normalize_query_text(values: pd.Series) -> pd.Series:
    """Минимальная нормализация для группировки формулировок."""
    return (
        values.astype("string")
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


train_query_groups = normalize_query_text(train["search_query"])
benchmark_query_groups = normalize_query_text(queries["search_query"])

print(f"Поисковых событий train: {len(train):,}")
print(f"Уникальных формулировок train: {train_query_groups.nunique():,}")
print(f"Поисковых событий benchmark: {len(queries):,}")
print(f"Уникальных формулировок benchmark: {benchmark_query_groups.nunique():,}")


In [ ]:
train_text_frequency = train_query_groups.value_counts()

benchmark_text_frequency = (
    benchmark_query_groups
    .map(train_text_frequency)
    .fillna(0)
    .astype(int)
)

print(
    "Benchmark-запросов с формулировкой из train: "
    f"{benchmark_text_frequency.gt(0).sum():,} / {len(queries):,} "
    f"({benchmark_text_frequency.gt(0).mean():.2%})"
)

display(
    benchmark_text_frequency
    .describe(percentiles=[0.5, 0.9, 0.99])
    .rename("Частота формулировки в train")
    .to_frame()
)


In [ ]:
def summarize_query_events(
    name: str,
    frame: pd.DataFrame,
    normalized_text: pd.Series,
) -> dict:
    return {
        "dataset": name,
        "events": len(frame),
        "query_length_median": normalized_text.str.len().median(),
        "query_length_p90": normalized_text.str.len().quantile(0.90),
        "category_0, %": frame["search_category"].eq(0).mean() * 100,
        "nonempty_params, %": (
            frame["search_infm_params_text"]
            .astype("string")
            .str.strip()
            .ne("")
            .mean() * 100
        ),
        "delivery_search, %": (
            frame["search_is_delivery_search"].ne(0).mean() * 100
        ),
    }


display(
    pd.DataFrame([
        summarize_query_events("train", train, train_query_groups),
        summarize_query_events(
            "benchmark", queries, benchmark_query_groups
        ),
    ]).round(2)
)


In [ ]:
train_unique_texts = train_query_groups.drop_duplicates()
benchmark_texts = benchmark_query_groups

print(
    "Benchmark-текстов, встречавшихся в train: "
    f"{benchmark_text_frequency.gt(0).sum():,} / {len(queries):,} "
    f"({benchmark_text_frequency.gt(0).mean():.2%})"
)

for name, texts in (
    ("train: строки", train_query_groups),
    ("train: уникальные тексты", train_unique_texts),
    ("benchmark", benchmark_texts),
):
    lengths = texts.str.len()
    print(
        f"{name}: n={len(texts):,}, "
        f"median={lengths.median():.0f}, "
        f"p90={lengths.quantile(0.90):.0f}, "
        f"p99={lengths.quantile(0.99):.0f}"
    )

bins = [0, 10, 15, 20, 25, 30, 40, 50, float("inf")]
labels = ["1–10", "11–15", "16–20", "21–25", "26–30", "31–40", "41–50", ">50"]

length_comparison = pd.DataFrame({
    "train_unique_texts": pd.cut(
        train_unique_texts.str.len(), bins=bins, labels=labels
    ).value_counts(normalize=True, sort=False),
    "benchmark": pd.cut(
        benchmark_texts.str.len(), bins=bins, labels=labels
    ).value_counts(normalize=True, sort=False),
}).mul(100)

display(length_comparison.round(1))

ax = length_comparison.plot.bar(figsize=(9, 3.5), rot=0)
ax.set_title("Длина уникальных формулировок: train и benchmark")
ax.set_xlabel("Символов в запросе")
ax.set_ylabel("Доля формулировок, %")
plt.tight_layout()
plt.show()

display(
    queries.loc[
        queries["search_category"].eq(0),
        ["search_query", "search_location_id", "search_infm_params_text"],
    ].head(10)
)


В train и benchmark близки длины уникальных формулировок, но различаются
заполненность параметров, доля категории `0` и распределение локаций.
Поэтому валидацию дополнительно выравниваем по наличию параметров.


## 5. Валидация на новых формулировках


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

SEED = 42


In [ ]:
def normalize_query_group(values: pd.Series) -> pd.Series:
    return (
        values.astype("string")
        .str.casefold()
        .str.replace("ё", "е", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


train_query_groups = normalize_query_group(train["search_query"])

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED,
)
fit_idx, validation_pool_idx = next(
    splitter.split(train, groups=train_query_groups)
)

fit_events = train.iloc[fit_idx].copy()
validation_pool = train.iloc[validation_pool_idx].copy()
fit_groups = set(train_query_groups.iloc[fit_idx])

validation_unique = (
    validation_pool
    .assign(
        query_group=train_query_groups.iloc[validation_pool_idx].to_numpy()
    )
    .sample(frac=1, random_state=SEED)
    .drop_duplicates("query_group")
    .copy()
)

validation_unique["has_params"] = (
    validation_unique["search_infm_params_text"]
    .astype("string")
    .str.strip()
    .ne("")
)
benchmark_has_params = (
    queries["search_infm_params_text"]
    .astype("string")
    .str.strip()
    .ne("")
)

n_with_params = int(benchmark_has_params.sum())
n_without_params = len(queries) - n_with_params

validation_events = pd.concat(
    [
        validation_unique.loc[validation_unique["has_params"]]
        .sample(n=n_with_params, random_state=SEED),
        validation_unique.loc[~validation_unique["has_params"]]
        .sample(n=n_without_params, random_state=SEED),
    ],
    ignore_index=True,
).sample(frac=1, random_state=SEED).reset_index(drop=True)

assert len(validation_events) == len(queries)
assert validation_events["query_group"].is_unique
assert set(validation_events["query_group"]).isdisjoint(fit_groups)

print(f"fit events:        {len(fit_events):,}")
print(f"validation events: {len(validation_events):,}")


In [ ]:
validation_comparison = pd.DataFrame([
    summarize_query_events(
        "validation",
        validation_events,
        normalize_query_text(validation_events["search_query"]),
    ),
    summarize_query_events(
        "benchmark",
        queries,
        benchmark_query_groups,
    ),
])

display(validation_comparison.round(2))

print(
    "Уникальных location_id — validation / benchmark: "
    f"{validation_events['search_location_id'].nunique():,} / "
    f"{queries['search_location_id'].nunique():,}"
)


In [ ]:
location_comparison = pd.DataFrame({
    "validation_pct": (
        validation_events["search_location_id"]
        .value_counts(normalize=True)
        .mul(100)
    ),
    "benchmark_pct": (
        queries["search_location_id"]
        .value_counts(normalize=True)
        .mul(100)
    ),
}).fillna(0)

top_benchmark_locations = (
    queries["search_location_id"]
    .value_counts()
    .head(15)
    .index
)

display(
    location_comparison
    .loc[top_benchmark_locations]
    .round(2)
)

for name, frame in (
    ("validation", validation_events),
    ("benchmark", queries),
):
    top_10_share = (
        frame["search_location_id"]
        .value_counts(normalize=True)
        .head(10)
        .sum()
    )
    print(f"{name}: доля 10 самых частых location_id = {top_10_share:.2%}")


Валидационный корпус содержит все таргеты и случайно выбранные объявления
train. После формирования он сортируется по `item_id`; положение таргета
не кодируется порядком строк.


In [ ]:
ITEM_COLUMNS = [column for column in train.columns if column.startswith("item_")]
CORPUS_SIZE = len(items)

train_items = (
    train[ITEM_COLUMNS]
    .drop_duplicates("item_id")
    .copy()
)

validation_target_ids = set(
    validation_events["item_id"].astype("string")
)

required_items = train_items.loc[
    train_items["item_id"].astype("string").isin(validation_target_ids)
]
other_items = train_items.loc[
    ~train_items["item_id"].astype("string").isin(validation_target_ids)
]

n_other = CORPUS_SIZE - len(required_items)
assert n_other >= 0 and len(other_items) >= n_other

# Состав корпуса по-прежнему содержит все таргеты, но его порядок не кодирует таргет.
validation_items = (
    pd.concat(
        [
            required_items,
            other_items.sample(n=n_other, random_state=SEED),
        ],
        ignore_index=True,
    )
    .assign(__stable_item_id=lambda frame: frame["item_id"].astype("string"))
    .sort_values("__stable_item_id", kind="mergesort")
    .drop(columns="__stable_item_id")
    .reset_index(drop=True)
)

validation_corpus_ids = set(validation_items["item_id"].astype("string"))

assert len(validation_items) == CORPUS_SIZE
assert validation_items["item_id"].is_unique
assert validation_target_ids <= validation_corpus_ids
assert all(column.startswith("item_") for column in validation_items.columns)

print(f"валидационный корпус: {len(validation_items):,}")
print(f"обязательных таргетов: {len(required_items):,}")
print("Порядок корпуса: stable item_id; search_* columns отсутствуют")


In [ ]:
corpus_comparison = pd.DataFrame({
    "validation_corpus_pct": (
        validation_items["item_microcat_id"]
        .value_counts(normalize=True)
        .mul(100)
    ),
    "benchmark_corpus_pct": (
        items["item_microcat_id"]
        .value_counts(normalize=True)
        .mul(100)
    ),
}).fillna(0)

corpus_comparison["difference_pp"] = (
    corpus_comparison["validation_corpus_pct"]
    - corpus_comparison["benchmark_corpus_pct"]
)

display(
    corpus_comparison
    .reindex(
        corpus_comparison["difference_pp"].abs()
        .sort_values(ascending=False)
        .index
    )
    .head(15)
    .round(2)
)


## Ограничения офлайн-оценки

В train для поискового события известен один выбранный item, поэтому
офлайн считаем `HitRate@50`. В закрытом benchmark у запроса может быть
несколько релевантных объявлений, а итоговая метрика — `Recall@50`.
Абсолютные значения напрямую не сопоставимы, но валидация подходит для
сравнения конфигураций на новых формулировках.
